# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [2]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [3]:
# Add as many cells as you need
df = pd.read_csv("zillow_cleaned.csv")

In [4]:
df.head()

,airconditioningtypeid,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedfinishedsquarefeet,fips,garagecarcnt,garagetotalsqft,heatingorsystemtypeid,latitude,...,regionidcity,regionidcounty,regionidneighborhood,regionidzip,roomcnt,unitcnt,yearbuilt,numberofstories,censustractandblock,taxvaluedollarcnt
0,1.0,3.5,4.0,6.4072,3100.0,6059.0,2.000000,633.00000,2.0,33634931.0,...,53571.0,1286.0,118849.0,96978.0,0.0,1.0,1998.0,1.0,6.059063e+13,1023282.0
1,1.0,1.0,2.0,6.4072,1465.0,6111.0,1.000000,0.00000,2.0,34449266.0,...,13091.0,2061.0,118849.0,97099.0,5.0,1.0,1967.0,1.0,6.111001e+13,464000.0
2,1.0,2.0,3.0,6.4072,1243.0,6059.0,2.000000,440.00000,2.0,33886168.0,...,21412.0,1286.0,118849.0,97078.0,6.0,1.0,1962.0,1.0,6.059022e+13,564778.0
3,1.0,3.0,4.0,8.0000,2376.0,6037.0,1.775842,330.01199,2.0,34245180.0,...,396551.0,3101.0,118849.0,96330.0,0.0,1.0,1970.0,1.0,6.037300e+13,145143.0
4,1.0,3.0,3.0,8.0000,1312.0,6037.0,1.775842,330.01199,2.0,34185120.0,...,12447.0,3101.0,268548.0,96451.0,0.0,1.0,1964.0,1.0,6.037124e+13,119407.0


In [5]:
y = df["taxvaluedollarcnt"]
y.head()

0    1023282.0
1     464000.0
2     564778.0
3     145143.0
4     119407.0
Name: taxvaluedollarcnt, dtype: float64

In [8]:
X = df.drop(columns="taxvaluedollarcnt")
X.head()

,airconditioningtypeid,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedfinishedsquarefeet,fips,garagecarcnt,garagetotalsqft,heatingorsystemtypeid,latitude,...,propertyzoningdesc,regionidcity,regionidcounty,regionidneighborhood,regionidzip,roomcnt,unitcnt,yearbuilt,numberofstories,censustractandblock
0,1.0,3.5,4.0,6.4072,3100.0,6059.0,2.000000,633.00000,2.0,33634931.0,...,1833.0,53571.0,1286.0,118849.0,96978.0,0.0,1.0,1998.0,1.0,6.059063e+13
1,1.0,1.0,2.0,6.4072,1465.0,6111.0,1.000000,0.00000,2.0,34449266.0,...,1833.0,13091.0,2061.0,118849.0,97099.0,5.0,1.0,1967.0,1.0,6.111001e+13
2,1.0,2.0,3.0,6.4072,1243.0,6059.0,2.000000,440.00000,2.0,33886168.0,...,1833.0,21412.0,1286.0,118849.0,97078.0,6.0,1.0,1962.0,1.0,6.059022e+13
3,1.0,3.0,4.0,8.0000,2376.0,6037.0,1.775842,330.01199,2.0,34245180.0,...,767.0,396551.0,3101.0,118849.0,96330.0,0.0,1.0,1970.0,1.0,6.037300e+13
4,1.0,3.0,3.0,8.0000,1312.0,6037.0,1.775842,330.01199,2.0,34185120.0,...,571.0,12447.0,3101.0,268548.0,96451.0,0.0,1.0,1964.0,1.0,6.037124e+13


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state, shuffle=True)

In [10]:
X_train.head()

,airconditioningtypeid,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedfinishedsquarefeet,fips,garagecarcnt,garagetotalsqft,heatingorsystemtypeid,latitude,...,propertyzoningdesc,regionidcity,regionidcounty,regionidneighborhood,regionidzip,roomcnt,unitcnt,yearbuilt,numberofstories,censustractandblock
49643,13.0,2.0,3.0,6.4072,1533.0,6059.0,2.000000,498.00000,2.0,33863052.0,...,1833.0,16764.0,1286.0,113455.0,97027.0,7.0,1.0,1975.0,1.0,6.059022e+13
51722,13.0,3.0,4.0,6.4072,2674.0,6059.0,2.000000,471.00000,2.0,33588349.0,...,1833.0,12773.0,1286.0,118849.0,96995.0,9.0,1.0,1980.0,2.0,6.059032e+13
34287,1.0,1.0,2.0,4.0000,873.0,6037.0,1.775842,330.01199,7.0,34304325.0,...,564.0,12447.0,3101.0,34213.0,96368.0,0.0,1.0,1955.0,1.0,6.037106e+13
47744,1.0,2.5,2.0,6.4072,1460.0,6059.0,2.000000,420.00000,6.0,33779089.0,...,1833.0,24832.0,1286.0,118849.0,97052.0,6.0,1.0,1974.0,2.0,6.059110e+13
18035,1.0,1.0,2.0,4.0000,676.0,6037.0,1.775842,330.01199,2.0,34483536.0,...,665.0,25621.0,3101.0,118849.0,97324.0,0.0,1.0,1981.0,1.0,6.037911e+13


In [11]:
y_train.head()

49643    343174.0
51722    853000.0
34287    103274.0
47744    415237.0
18035     37129.0
Name: taxvaluedollarcnt, dtype: float64

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [3]:
# Add as many cells as you need


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

> Your text here

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [4]:
# Add as many cells as you need


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




> Your text here

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [5]:
# Add as many cells as you need


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


> Your text here

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [6]:
# Add as many cells as you need


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [7]:
# Add as many cells as you need


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here